# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [1]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [2]:
import numpy as np
from itertools import combinations
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [3]:
def build_classifiers():
    models = [
        LinearRegression(),
        KNeighborsClassifier(),
        SVC(kernel="linear"),
        DecisionTreeClassifier(),
        GaussianNB(),
        QuadraticDiscriminantAnalysis(),
    ]

    for model in models:
        model.fit(data_set, labels.ravel())

    best_models = None
    best_accuracy = -1.0

    for combo in combinations(models, 3):
        predicted = build_stacked_classifier(combo)
        accuracy = accuracy_score(test_labels, predicted)

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_models = list(combo)

    return best_models

In [ ]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))
    
    # stacked classifier part:
    stacked_classifier = GaussianNB()
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [5]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

1.0


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set, labels = make_classification(
        n_samples=sample_number, n_features=feature_number,
        n_informative=feature_number, n_redundant=0,
        n_classes=label_number, random_state=42)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, test_set, train_labels, test_labels = train_test_split(
        *generate_data(train_set_size + test_set_size, dimension, labels),
        test_size=test_set_size, random_state=42)


# init weights
number_of_iterations = 10
weights = np.ones((train_set_size,)) / train_set_size


def train_model(classifier, weights):
    model = clone(classifier)
    return model.fit(X=train_set, y=train_labels, sample_weight=weights)

def calculate_accuracy_vector(predicted, labels):
    result = []
    for i in range(len(predicted)):
        result.append(int(not predicted[i] == labels[i]))
    return result

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [7]:
def set_new_weights(model):
    I = np.array(calculate_accuracy_vector(model.predict(train_set), train_labels))
    return (1 + I) / np.sum(1 + I)

Train the classifier with the code below:

In [8]:
classifier = DecisionTreeClassifier(max_depth=3, random_state=1)
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00018308 0.00018308 0.00018308 ... 0.00018308 0.00018308 0.00018308]


Set the validation data set:

In [9]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [10]:
def get_prediction(x):
    output = np.array([classifier.predict(x) for classifier in classifiers])
    return [np.argmax(np.bincount(output[:, i])) for i in range(len(x))]

Test it:

In [11]:
from sklearn.metrics import accuracy_score

single_pred = classifiers[0].predict(test_set)
voting_pred = get_prediction(test_set)
print("Single accuracy:", accuracy_score(test_labels, single_pred))
print("Voting accuracy:", accuracy_score(test_labels, voting_pred))

Single accuracy: 0.906
Voting accuracy: 0.911
